# Dynamic-k Model Matrix Comparison

Run this notebook top to bottom in the online IDE. It compares the project baselines against `latent_regime_k` for the requested drafter-target model pairs and saves one CSV plus one combined PNG per pair.

Required matrix by default:

- 70M drafter -> 1.5B, 3B, 7B targets
- 125M drafter -> 1.5B, 3B, 7B targets

Optional 14B and 32B targets are controlled by `INCLUDE_LARGE_TARGETS`.

In [ ]:
from __future__ import annotations

import csv
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

# Set this only if your notebook kernel starts outside the repository.
REPO_ROOT_OVERRIDE = ""


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "main.py").exists() and (candidate / "research").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Set REPO_ROOT_OVERRIDE.")


REPO_ROOT = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "research" / "v.poponnikov" / "notebooks" / "dynamic_k_comparison.py"
RESULT_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "results" / "model_matrix"
PLOTS_BASE_DIR = REPO_ROOT / "research" / "v.poponnikov" / "plots"
EXPERIMENTS = ["01_baseline", "08_+speedup_adapt", "latent_regime_k"]

print(f"Repository: {REPO_ROOT}")
print(f"Comparison script: {SCRIPT}")
print(f"Results base: {RESULT_BASE_DIR}")
print(f"Plots base: {PLOTS_BASE_DIR}")

## Helper for Cell-Based Runs

This helper runs Python commands with `PYTHONPATH=src` and streams output into the notebook cell.

In [ ]:
def run_python(*args: object) -> None:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src")
    env.setdefault("HF_HOME", str(REPO_ROOT / ".hf-cache"))
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("USE_TF", "0")
    env.setdefault("USE_FLAX", "0")
    env.setdefault("TRANSFORMERS_NO_TF", "1")
    env.setdefault("TRANSFORMERS_NO_FLAX", "1")
    env.setdefault("TRANSFORMERS_NO_TORCHVISION", "1")

    command = [sys.executable, *[str(arg) for arg in args]]
    print("Running:", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")

## Optional Dependency Install

Run this once on a fresh online IDE image. After it finishes, restart the notebook kernel and rerun the setup/helper cells with `INSTALL_DEPENDENCIES = False`.

In [ ]:
INSTALL_DEPENDENCIES = True

if INSTALL_DEPENDENCIES:
    run_python(
        "-m",
        "pip",
        "install",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
        "torch>=2.5,<2.11",
    )
    runtime_packages = [
        "typer>=0.15",
        "rich>=13",
        "tqdm>=4.68",
        "transformers>=4.48,<5",
        "accelerate>=1.3",
        "bitsandbytes>=0.45",
        "safetensors>=0.5",
        "datasets>=3,<5",
        "pandas>=2.2",
        "numpy<2",
        "scipy>=1.14",
        "scikit-learn>=1.6",
        "tokenizers>=0.21",
        "regex>=2024",
        "pyahocorasick>=2.3",
        "networkx>=3.4",
        "pydantic>=2.10",
        "python-dotenv>=1.2",
        "matplotlib>=3.10",
        "seaborn>=0.13",
    ]
    run_python("-m", "pip", "install", *runtime_packages)
    print("Dependency install finished. Restart the notebook kernel once, then set INSTALL_DEPENDENCIES = False.")
else:
    print("Dependency install skipped.")

## GPU Check

In [ ]:
try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print("GPU:", props.name)
        print("VRAM GB:", round(props.total_memory / 1024**3, 2))
except ModuleNotFoundError:
    print("Torch is not installed in this kernel. Enable the dependency install cell above.")

## Confirm Research Experiments Are Discoverable

In [ ]:
run_python(REPO_ROOT / "src" / "main.py", "--list", "--research")

## Tiny Sanity Run

This optional small run checks that the runner can load models, run all three techniques, and write outputs before the full matrix starts.

In [ ]:
RUN_TINY_SANITY = True

if RUN_TINY_SANITY:
    run_python(
        SCRIPT,
        "--output-dir",
        RESULT_BASE_DIR / "tiny_sanity",
        "--plots-dir",
        PLOTS_BASE_DIR / "tiny_sanity",
        "--tiny",
        "--samples",
        2,
        "--max-new-tokens",
        16,
        "--device",
        "cuda",
    )
else:
    print("Tiny sanity run skipped.")

## Required Model Matrix

This is the main benchmark. It runs:

- `EleutherAI/pythia-70m` as drafter against Qwen 1.5B, 3B, and 7B targets
- `facebook/opt-125m` as drafter against Qwen 1.5B, 3B, and 7B targets

Outputs per pair:

- `research/v.poponnikov/results/model_matrix/<pair>/metrics.csv`
- `research/v.poponnikov/plots/<pair>-plots/comparison.png`

Set `INCLUDE_LARGE_TARGETS = True` only if there is enough memory and time for 14B and 32B targets.

In [ ]:
RUN_MODEL_MATRIX = True
MATRIX_SAMPLES = 50
MATRIX_MAX_NEW_TOKENS = 128
MATRIX_DRAFTS = ["70m", "125m"]
MATRIX_TARGETS = ["1.5b", "3b", "7b"]
INCLUDE_LARGE_TARGETS = False
CONTINUE_ON_ERROR = False

if RUN_MODEL_MATRIX:
    command = [
        SCRIPT,
        "--matrix",
        "--output-dir",
        RESULT_BASE_DIR,
        "--plots-dir",
        PLOTS_BASE_DIR,
        "--samples",
        MATRIX_SAMPLES,
        "--max-new-tokens",
        MATRIX_MAX_NEW_TOKENS,
        "--device",
        "cuda",
        "--draft-sizes",
        *MATRIX_DRAFTS,
        "--target-sizes",
        *MATRIX_TARGETS,
    ]
    if INCLUDE_LARGE_TARGETS:
        command.append("--include-large-targets")
    if CONTINUE_ON_ERROR:
        command.append("--continue-on-error")
    run_python(*command)
else:
    print("Model matrix skipped.")